# 🧱 Lab: Triage CSV Rows with an AI Pipeline

**Module: Ship a Working AI Feature Today** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Run** a complete AI pipeline end to end that reads a CSV, classifies each row with an LLM, and writes results
2. **Apply** the six pipeline stages to a real task — customer-support ticket triage
3. **Identify** which stage breaks when something goes wrong and fix it
4. **Debug** realistic pipeline failures by locating them in the correct stage

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| AI Pipeline Stages | A practical AI feature pipeline with six boxes: ingest, prepare, call, validate, log, evaluate. The model call is just one box, not the whole system. | mini-map-pipeline-stages |
| End-to-End Pipeline Execution | — | This lab |
| Failure Localization | — | This lab |

## Prerequisites

- **mini-map-pipeline-stages**: The six stages of an AI pipeline (Ingest → Prepare → Prompt → Call Model → Parse Output → Deliver)


## Setup

We'll use the OpenAI SDK to call an LLM. Make sure you have your API key in a `.env` file.

In [1]:
import os
import csv
import json
from io import StringIO
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## The Task: Customer Support Ticket Triage

You're building an AI feature that reads customer support tickets from a CSV and classifies each one into a category: **Billing**, **Technical**, **Account**, or **General**. This is exactly the kind of feature an AI engineer ships in a real product.

We'll build this as a six-stage pipeline, then deliberately break it at different stages to practice failure localization.

```
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│ 1.INGEST │──▶│2.PREPARE │──▶│ 3.PROMPT │──▶│ 4.CALL   │──▶│ 5.PARSE  │──▶│6.DELIVER │
│ Read CSV │   │Clean rows│   │Build msg │   │  MODEL   │   │ Extract  │   │Write CSV │
└──────────┘   └──────────┘   └──────────┘   └──────────┘   └──────────┘   └──────────┘
```

## Stage 1 — Ingest: Create and Load the CSV

First, we simulate a small CSV of support tickets. In production this would be a database query or file upload.

In [2]:
# Create a small CSV of support tickets
CSV_DATA = """ticket_id,customer,message
101,Alice,"I was charged twice for my subscription last month. Please refund the duplicate."
102,Bob,"The app crashes every time I try to upload a file larger than 10MB."
103,Carol,"How do I change the email address on my account?"
104,Dan,"I love your product! Just wanted to say thanks to the team."
105,Eve,"My invoice shows a charge for a plan I never signed up for."
106,Frank,"Getting a 502 error on the API endpoint since this morning."
"""

os.makedirs("data", exist_ok=True)
csv_path = os.path.join("data", "tickets.csv")

with open(csv_path, "w", newline="") as f:
    f.write(CSV_DATA.strip())

print(f"✅ Wrote {csv_path}")

✅ Wrote data\tickets.csv


In [3]:
# Stage 1: Ingest — read the CSV into a list of dicts
def ingest(path):
    """Stage 1: Load tickets from a CSV file."""
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        tickets = list(reader)
    print(f"Stage 1 (Ingest): Loaded {len(tickets)} tickets from {path}")
    return tickets

tickets = ingest(csv_path)
tickets[0]

Stage 1 (Ingest): Loaded 6 tickets from data\tickets.csv


{'ticket_id': '101',
 'customer': 'Alice',
 'message': 'I was charged twice for my subscription last month. Please refund the duplicate.'}

## Stage 2 — Prepare: Clean and Validate Rows

Before sending anything to a model, we validate that each row has the fields we need and the message isn't empty.

In [4]:
# Stage 2: Prepare — validate and clean each ticket
def prepare(tickets):
    """Stage 2: Validate tickets and strip whitespace."""
    clean = []
    for t in tickets:
        msg = t.get("message", "").strip()
        if not msg:
            print(f"  ⚠️  Skipping ticket {t.get('ticket_id', '?')}: empty message")
            continue
        clean.append({"ticket_id": t["ticket_id"], "customer": t["customer"], "message": msg})
    print(f"Stage 2 (Prepare): {len(clean)} tickets ready (out of {len(tickets)})")
    return clean

clean_tickets = prepare(tickets)
print(f"First clean ticket: {clean_tickets[0]}")

Stage 2 (Prepare): 6 tickets ready (out of 6)
First clean ticket: {'ticket_id': '101', 'customer': 'Alice', 'message': 'I was charged twice for my subscription last month. Please refund the duplicate.'}


## Stage 3 — Prompt: Build the Classification Prompt

We construct a system prompt that tells the model exactly what to do, and a user prompt for each ticket.

In [5]:
SYSTEM_PROMPT = """You are a support ticket classifier. For each customer message, respond with ONLY a JSON object:
{"category": "<one of: Billing, Technical, Account, General>", "confidence": "<high, medium, or low>"}

Rules:
- Billing: charges, refunds, invoices, payments
- Technical: bugs, errors, crashes, API issues
- Account: email, password, profile, settings changes
- General: everything else (feedback, questions, greetings)
"""

def build_prompt(ticket):
    """Stage 3: Assemble messages for the LLM."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {"role": "user", "content": f"Classify this ticket:\n\n{ticket['message']}"}
    ]

# Preview the prompt for the first ticket
sample_messages = build_prompt(clean_tickets[0])
print("Stage 3 (Prompt): Built messages for ticket", clean_tickets[0]["ticket_id"])
print(f"  System: {sample_messages[0]['content'][:80]}...")
print(f"  User:   {sample_messages[1]['content']}")

Stage 3 (Prompt): Built messages for ticket 101
  System: You are a support ticket classifier. For each customer message, respond with ONL...
  User:   Classify this ticket:

I was charged twice for my subscription last month. Please refund the duplicate.


## Stage 4 — Call Model: Send to the LLM

We call the OpenAI API for each ticket. In production you'd batch these or use async, but for learning we keep it simple.

In [6]:
def call_model(messages):
    """Stage 4: Send prompt to LLM and return raw response text."""
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content

# Test with first ticket
raw_output = call_model(sample_messages)
print("Stage 4 (Call Model): Raw response →", raw_output)

Stage 4 (Call Model): Raw response → {"category": "Billing", "confidence": "high"}


## Stage 5 — Parse Output: Extract Structured Data

The model returns a JSON string. We parse it into a Python dict. This is where many real pipelines break — the model might return extra text or malformed JSON.

In [7]:
def parse_output(raw_text):
    """Stage 5: Parse the model's JSON response."""
    result = json.loads(raw_text)
    # Validate expected fields exist
    assert "category" in result, "Missing 'category' field"
    assert "confidence" in result, "Missing 'confidence' field"
    return result

parsed = parse_output(raw_output)
print("Stage 5 (Parse Output):", parsed)

Stage 5 (Parse Output): {'category': 'Billing', 'confidence': 'high'}


## Stage 6 — Deliver: Collect Results

In a real app you'd write to a database, send a webhook, or update a UI. Here we collect results into a list for display.

In [8]:
def deliver(ticket, classification):
    """Stage 6: Format the final result for delivery."""
    return {
        "ticket_id": ticket["ticket_id"],
        "customer": ticket["customer"],
        "category": classification["category"],
        "confidence": classification["confidence"],
    }

# Test with first ticket
result = deliver(clean_tickets[0], parsed)
print("Stage 6 (Deliver):", result)

Stage 6 (Deliver): {'ticket_id': '101', 'customer': 'Alice', 'category': 'Billing', 'confidence': 'high'}


## Run the Full Pipeline

Now we chain all six stages together and process every ticket. This is the end-to-end run.

In [9]:
def run_pipeline(csv_path):
    """Run the full six-stage triage pipeline."""
    # Stage 1: Ingest
    tickets = ingest(csv_path)
    
    # Stage 2: Prepare
    clean = prepare(tickets)
    
    results = []
    for ticket in clean:
        # Stage 3: Prompt
        messages = build_prompt(ticket)
        
        # Stage 4: Call Model
        raw = call_model(messages)
        
        # Stage 5: Parse Output
        classification = parse_output(raw)
        
        # Stage 6: Deliver
        results.append(deliver(ticket, classification))
    
    return results

results = run_pipeline(csv_path)

# Display results as a table
header = "| Ticket | Customer | Category | Confidence |\n|--------|----------|----------|------------|\n"
rows = ""
for r in results:
    rows += f"| {r['ticket_id']} | {r['customer']} | {r['category']} | {r['confidence']} |\n"

md("## ✅ Pipeline Results\n\n" + header + rows)

Stage 1 (Ingest): Loaded 6 tickets from data\tickets.csv
Stage 2 (Prepare): 6 tickets ready (out of 6)


## ✅ Pipeline Results

| Ticket | Customer | Category | Confidence |
|--------|----------|----------|------------|
| 101 | Alice | Billing | high |
| 102 | Bob | Technical | high |
| 103 | Carol | Account | high |
| 104 | Dan | General | high |
| 105 | Eve | Billing | high |
| 106 | Frank | Technical | high |


## Break It: Failure Localization Practice

The real skill of working with AI pipelines is knowing **which stage broke**. We'll deliberately introduce failures at different stages, catch them, and name the stage.

We'll use a wrapper function that catches errors and maps them to pipeline stages.

In [10]:
def diagnose_stage(stage_num, stage_name, error):
    """Format a stage-failure diagnosis."""
    return (
        f"💥 **Pipeline broke at Stage {stage_num} — {stage_name}**\n\n"
        f"Error type: `{type(error).__name__}`\n\n"
        f"Message: `{error}`\n\n"
        f"*Fix:* Look at your **{stage_name}** code, not the model or the prompt."
    )

### Break Stage 1 — Ingest

What happens when the file doesn't exist?

In [11]:
# Deliberately pass a wrong file path
try:
    ingest("data/nonexistent.csv")
except FileNotFoundError as e:
    md(diagnose_stage(1, "Ingest", e))

💥 **Pipeline broke at Stage 1 — Ingest**

Error type: `FileNotFoundError`

Message: `[Errno 2] No such file or directory: 'data/nonexistent.csv'`

*Fix:* Look at your **Ingest** code, not the model or the prompt.

### Break Stage 2 — Prepare

What happens when a ticket has a missing field?

In [12]:
# A ticket with no 'message' key at all
bad_tickets = [{"ticket_id": "999", "customer": "Ghost"}]

result = prepare(bad_tickets)
md(
    f"📍 **Stage 2 — Prepare** handled the problem gracefully.\n\n"
    f"The ticket with no message was skipped: {len(result)} tickets passed through."
)

  ⚠️  Skipping ticket 999: empty message
Stage 2 (Prepare): 0 tickets ready (out of 1)


📍 **Stage 2 — Prepare** handled the problem gracefully.

The ticket with no message was skipped: 0 tickets passed through.

### Break Stage 3 — Prompt

What happens when you reference a field that doesn't exist in the ticket dict?

In [13]:
# A build_prompt that references a field that doesn't exist
def build_prompt_broken(ticket):
    """A broken version that references a non-existent field."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {"role": "user", "content": f"Classify: {ticket['subject']}"} # 'subject' doesn't exist!
    ]

try:
    build_prompt_broken(clean_tickets[0])
except KeyError as e:
    md(diagnose_stage(3, "Prompt", e))

💥 **Pipeline broke at Stage 3 — Prompt**

Error type: `KeyError`

Message: `'subject'`

*Fix:* Look at your **Prompt** code, not the model or the prompt.

### Break Stage 5 — Parse Output

What happens when the model returns something that isn't valid JSON?

In [14]:
# Simulate a model returning malformed output
bad_response = "I'd classify this as Billing with high confidence."

try:
    parse_output(bad_response)
except json.JSONDecodeError as e:
    md(diagnose_stage(5, "Parse Output", e))

💥 **Pipeline broke at Stage 5 — Parse Output**

Error type: `JSONDecodeError`

Message: `Expecting value: line 1 column 1 (char 0)`

*Fix:* Look at your **Parse Output** code, not the model or the prompt.

## Pipeline with Stage Labels

Let's build a hardened version of the pipeline that catches errors at each stage and tells you exactly where it broke. This is the pattern you'll use in real AI engineering work.

In [15]:
def run_pipeline_safe(csv_path):
    """Run the pipeline with per-stage error handling."""
    # Stage 1
    try:
        tickets = ingest(csv_path)
    except Exception as e:
        md(diagnose_stage(1, "Ingest", e))
        return []

    # Stage 2
    try:
        clean = prepare(tickets)
    except Exception as e:
        md(diagnose_stage(2, "Prepare", e))
        return []

    results = []
    for ticket in clean:
        tid = ticket["ticket_id"]
        try:
            messages = build_prompt(ticket)          # Stage 3
            raw = call_model(messages)               # Stage 4
            classification = parse_output(raw)       # Stage 5
            results.append(deliver(ticket, classification))  # Stage 6
        except KeyError as e:
            md(f"Ticket {tid}: " + diagnose_stage(3, "Prompt", e))
        except json.JSONDecodeError as e:
            md(f"Ticket {tid}: " + diagnose_stage(5, "Parse Output", e))
        except Exception as e:
            md(f"Ticket {tid}: " + diagnose_stage(4, "Call Model", e))

    return results

# Run the safe pipeline on a bad path to see Stage 1 diagnosis
run_pipeline_safe("data/doesnt_exist.csv")
print()  # blank line

# Run it on the real file — should succeed
safe_results = run_pipeline_safe(csv_path)
print(f"\n✅ Successfully triaged {len(safe_results)} tickets.")

💥 **Pipeline broke at Stage 1 — Ingest**

Error type: `FileNotFoundError`

Message: `[Errno 2] No such file or directory: 'data/doesnt_exist.csv'`

*Fix:* Look at your **Ingest** code, not the model or the prompt.


Stage 1 (Ingest): Loaded 6 tickets from data\tickets.csv
Stage 2 (Prepare): 6 tickets ready (out of 6)



✅ Successfully triaged 6 tickets.


## Review: Which Stage Was It?

Let's do a final review — given a set of error messages, can you name the stage? This is the core skill from `mini-map-pipeline-stages` applied to the real pipeline you just built.

In [16]:
quiz = [
    ("PermissionError: cannot read 'data/tickets.csv'", 1, "Ingest"),
    ("A ticket's 'message' field is empty so it gets skipped", 2, "Prepare"),
    ("KeyError: 'priority' when building the user prompt", 3, "Prompt"),
    ("openai.RateLimitError: you've exceeded your quota", 4, "Call Model"),
    ("json.JSONDecodeError: the model said 'Billing' instead of JSON", 5, "Parse Output"),
    ("The downstream dashboard expects 'type' but we send 'category'", 6, "Deliver"),
]

output = "## Stage Identification Review\n\n"
output += "| Error | Stage | Name |\n|-------|-------|------|\n"
for desc, num, name in quiz:
    output += f"| {desc} | {num} | **{name}** |\n"

md(output)

## Stage Identification Review

| Error | Stage | Name |
|-------|-------|------|
| PermissionError: cannot read 'data/tickets.csv' | 1 | **Ingest** |
| A ticket's 'message' field is empty so it gets skipped | 2 | **Prepare** |
| KeyError: 'priority' when building the user prompt | 3 | **Prompt** |
| openai.RateLimitError: you've exceeded your quota | 4 | **Call Model** |
| json.JSONDecodeError: the model said 'Billing' instead of JSON | 5 | **Parse Output** |
| The downstream dashboard expects 'type' but we send 'category' | 6 | **Deliver** |


## 🎯 Summary

### What You Built

You built a complete AI-powered ticket triage pipeline that reads customer support tickets from a CSV, classifies each one via an LLM, and collects structured results — then you deliberately broke it at multiple stages and practiced diagnosing exactly where each failure occurred.

### Key Takeaways

1. **End-to-end pipeline** — even a simple AI feature like ticket classification touches all six stages: Ingest → Prepare → Prompt → Call Model → Parse Output → Deliver
2. **Stage isolation** — wrapping each stage in its own function makes failures easy to locate and fix independently
3. **Parse Output is fragile** — the model can return text that looks right but isn't valid JSON; this is the most common real-world pipeline breakage
4. **Failure localization** — when something breaks, asking "which of the six stages broke?" is faster than staring at the whole pipeline